In [0]:
dbutils.fs.ls('mnt/silver/Sales')

Out[1]: [FileInfo(path='dbfs:/mnt/silver/Sales/CountryRegionCurrency/', name='CountryRegionCurrency/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/mnt/silver/Sales/CreditCard/', name='CreditCard/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/mnt/silver/Sales/Currency/', name='Currency/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/mnt/silver/Sales/CurrencyRate/', name='CurrencyRate/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/mnt/silver/Sales/Customer/', name='Customer/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/mnt/silver/Sales/PersonCreditCard/', name='PersonCreditCard/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/mnt/silver/Sales/SalesOrderDetail/', name='SalesOrderDetail/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/mnt/silver/Sales/SalesOrderHeader/', name='SalesOrderHeader/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/mnt/silver/Sales/SalesOrderHeaderSalesReason/', name='SalesOrderHeaderSalesReason/', size=0, modific

In [0]:
dbutils.fs.ls('mnt/gold')

Out[2]: []

In [0]:
input_path = '/mnt/silver/Sales/Customer/'

In [0]:
df = spark.read.format('delta').load(input_path)

In [0]:
df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- PersonID: integer (nullable = true)
 |-- StoreID: integer (nullable = true)
 |-- TerritoryID: integer (nullable = true)
 |-- AccountNumber: string (nullable = true)
 |-- rowguid: string (nullable = true)
 |-- ModifiedDate: string (nullable = true)



In [0]:
display(df)

CustomerID,PersonID,StoreID,TerritoryID,AccountNumber,rowguid,ModifiedDate
1,null,934,1,AW00000001,3f5ae95e-b87d-4aed-95b4-c3797afcb74f,2014-09-12
2,null,1028,1,AW00000002,e552f657-a9af-4a7d-a645-c429d6e02491,2014-09-12
3,null,642,4,AW00000003,130774b1-db21-4ef3-98c8-c104bcd6ed6d,2014-09-12
4,null,932,4,AW00000004,ff862851-1daa-4044-be7c-3e85583c054d,2014-09-12
5,null,1026,4,AW00000005,83905bdc-6f5e-4f71-b162-c98da069f38a,2014-09-12
6,null,644,4,AW00000006,1a92df88-bfa2-467d-bd54-fcb9e647fdd7,2014-09-12
7,null,930,1,AW00000007,03e9273e-b193-448e-9823-fe0c44aeed78,2014-09-12
8,null,1024,5,AW00000008,801368b1-4323-4bfa-8bea-5b5b1e4bd4a0,2014-09-12
9,null,620,5,AW00000009,b900bb7f-23c3-481d-80da-c49a5bd6f772,2014-09-12
10,null,928,6,AW00000010,cdb6698d-2ff1-4fba-8f22-60ad1d11dabd,2014-09-12


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace

In [0]:
colList = df.columns

for old_col_name in colList:
    new_col_name = "".join(["_" + char if char.isupper() and not old_col_name[i-1].isupper() else char for i,char in enumerate(old_col_name)]).lstrip("_")

    df = df.withColumnRenamed(old_col_name, new_col_name)

In [0]:
display(df)

Customer_ID,Person_ID,Store_ID,Territory_ID,Account_Number,rowguid,Modified_Date
1,null,934,1,AW00000001,3f5ae95e-b87d-4aed-95b4-c3797afcb74f,2014-09-12
2,null,1028,1,AW00000002,e552f657-a9af-4a7d-a645-c429d6e02491,2014-09-12
3,null,642,4,AW00000003,130774b1-db21-4ef3-98c8-c104bcd6ed6d,2014-09-12
4,null,932,4,AW00000004,ff862851-1daa-4044-be7c-3e85583c054d,2014-09-12
5,null,1026,4,AW00000005,83905bdc-6f5e-4f71-b162-c98da069f38a,2014-09-12
6,null,644,4,AW00000006,1a92df88-bfa2-467d-bd54-fcb9e647fdd7,2014-09-12
7,null,930,1,AW00000007,03e9273e-b193-448e-9823-fe0c44aeed78,2014-09-12
8,null,1024,5,AW00000008,801368b1-4323-4bfa-8bea-5b5b1e4bd4a0,2014-09-12
9,null,620,5,AW00000009,b900bb7f-23c3-481d-80da-c49a5bd6f772,2014-09-12
10,null,928,6,AW00000010,cdb6698d-2ff1-4fba-8f22-60ad1d11dabd,2014-09-12


# Doing transformation to all tables

In [0]:
table_names = []

for i in dbutils.fs.ls('mnt/silver/Sales/'):
    table_names.append(i.name.split('/')[0])

table_names

Out[22]: ['CountryRegionCurrency',
 'CreditCard',
 'Currency',
 'CurrencyRate',
 'Customer',
 'PersonCreditCard',
 'SalesOrderDetail',
 'SalesOrderHeader',
 'SalesOrderHeaderSalesReason',
 'SalesPerson',
 'SalesPersonQuotaHistory',
 'SalesReason',
 'SalesTaxRate',
 'SalesTerritory',
 'SalesTerritoryHistory',
 'ShoppingCartItem',
 'SpecialOffer',
 'SpecialOfferProduct',
 'Store']

In [0]:
for name in table_names:
    path = '/mnt/silver/Sales/' + name
    df = spark.read.format('delta').load(path)
    colList = df.columns

    for old_col_name in colList:
        new_col_name = "".join(["_" + char if char.isupper() and not old_col_name[i-1].isupper() else char for i,char in enumerate(old_col_name)]).lstrip("_")

        df = df.withColumnRenamed(old_col_name, new_col_name)
    
    output_path = '/mnt/gold/Sales/' + name + '/'
    df.write.format('delta').mode('overwrite').save(output_path)

In [0]:
display(df)

Business_Entity_ID,Name,Sales_Person_ID,Demographics,rowguid,Modified_Date
292,Next-Door Bike Store,279,80000080000United SecurityBM1996Mountain210002ISDN13,a22517e3-848d-4ebe-b9d9-7437f3432304,2014-09-12
294,Professional Sales and Service,276,80000080000International BankBM1991Touring180004+T114,b50ca50b-c601-4a13-b07e-2c63862d71b4,2014-09-12
296,Riders Company,277,80000080000Primary Bank & ReserveBM1999Road210002DSL15,337c3688-1339-4e1a-a08a-b54b23566e49,2014-09-12
298,The Bike Mechanics,275,80000080000International SecurityBM1994Mountain180002DSL16,7894f278-f0c8-4d16-bd75-213fdbf13023,2014-09-12
300,Nationwide Supply,286,80000080000Guardian BankBM1987Touring210004+DSL17,c3fc9705-a8c4-4f3a-9550-eb2fa4b7b64d,2014-09-12
302,Area Bike Accessories,281,30000030000International BankBM1982Road9000AWT28,368be6dd-30e5-49bb-9a86-71fd49c58f4e,2014-09-12
304,Bicycle Accessories and Kits,283,30000030000Primary Bank & ReserveBM1990Mountain7000AWT19,35f40636-5105-49d5-869e-27e231189150,2014-09-12
306,Clamps & Brackets Co.,275,80000080000International SecurityBM1985Mountain170004+DSL10,64d06bfc-d060-405c-8c60-c067fe7c67df,2014-09-12
308,Valley Bicycle Specialists,277,3000000300000Primary Bank & ReserveOS1979Mountain720004+DSL66,59386b0c-652e-4668-b44b-4e1711793330,2014-09-12
310,New Bikes Company,279,1500000150000International SecurityOS1974Road390004+T140,47e4b6bd-5cd1-45a3-a231-79d930381c56,2014-09-12
